In [4]:
!pip install pyodbc pandas


In [3]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

# 1. Define server properties
server = 'MSITHINA15\\MSSQL'  
database = 'vgames'        

# 2. Build the standard connection string
params = urllib.parse.quote_plus(
    f'Driver={{SQL Server}};Server={server};Database={database};Trusted_Connection=yes;'
)

# 3. Create the modern SQLAlchemy Engine that Pandas expects
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# 4. Pull your raw table directly—Zero warnings, zero errors!
df = pd.read_sql_query("SELECT * FROM dbo.vgsales;", engine)

print(f"🚀 Success! Pristine dataset loaded with zero warnings.")
print(f"Total rows in your local Pandas Sandbox: {len(df)}")

# 5. Display the first 5 rows immediately in the same cell
df.head()


C:\Users\kushw\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1125.2'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


🚀 Success! Pristine dataset loaded with zero warnings.
Total rows in your local Pandas Sandbox: 16598


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.490002,29.02,3.77,8.46,82.739998
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.080000,3.58,6.81,0.77,40.240002
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.850000,12.88,3.79,3.31,35.820000
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.750000,11.01,3.28,2.96,33.000000
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.270000,8.89,10.22,1.00,31.370001


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16598 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


In [5]:
df.columns


Index(['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales'],
      dtype='object')

In [ ]:
import matplotlib.pyplot as plt
for col in sales_col:
    plt.boxplot(df[col])
    plt.show()


In [7]:
import pandas as pd

# Define your list of sales columns explicitly so it doesn't try to calculate quantiles on text columns
sales_col = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Global_Sales']  # <-- Update this with your exact sales column names

summary_data = []

# Loop ONLY through the numeric sales columns
for col in sales_col:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    IQR = q3 - q1
    lb = q1 - 1.5 * IQR
    ub = q3 + 1.5 * IQR
    
    mega_hits = df[df[col] > ub]
    standard_market_df = df[df[col] <= ub]
    
    # Safety Check: Only pull top attributes if mega_hits actually exist
    if len(mega_hits) > 0:
        top_genre = mega_hits['Genre'].value_counts().index[0]
        genre_pct = (mega_hits['Genre'].value_counts().iloc[0] / len(mega_hits)) * 100
        
        top_platform = mega_hits['Platform'].value_counts().index[0]
        platform_pct = (mega_hits['Platform'].value_counts().iloc[0] / len(mega_hits)) * 100
        
        top_year = mega_hits['Year'].value_counts().index[0]
    else:
        top_genre, genre_pct = "None", 0.0
        top_platform, platform_pct = "None", 0.0
        top_year = 0

    # Append data cleanly (Duplicate key removed)
    summary_data.append({
        "Sales_Region": col,
        "Upper_Bound_Threshold": round(ub, 2),
        "Mega_Hit_Count": len(mega_hits),
        "Standard_Game_Count": len(standard_market_df),
        "Mega_Hit_Mean_Sales": round(mega_hits[col].mean(), 2) if len(mega_hits) > 0 else 0,
        "Standard_Game_Mean_Sales": round(standard_market_df[col].mean(), 2),
        "Dominant_Genre": f"{top_genre} ({genre_pct:.1f}%)",
        "Dominant_Platform": f"{top_platform} ({platform_pct:.1f}%)",
        "Peak_Hit_Year": int(top_year)
    })

# 3. CONVERT TO DATAFRAME 
sales_summary_df = pd.DataFrame(summary_data)

# 4. Save it as a CSV for your Power BI Report
sales_summary_df.to_excel("sales_segments_summary.xlsx", index=False)

print("Saved as a true Excel file successfully!")



Saved as a true Excel file successfully!


In [8]:
sales_summary_df.shape

(4, 9)

In [9]:
sales_summary_df.head()

,Sales_Region,Upper_Bound_Threshold,Mega_Hit_Count,Standard_Game_Count,Mega_Hit_Mean_Sales,Standard_Game_Mean_Sales,Dominant_Genre,Dominant_Platform,Peak_Hit_Year
0,NA_Sales,0.60,1711,14887,1.60,0.11,Action (20.5%),PS2 (16.0%),2008
1,EU_Sales,0.27,2081,14517,0.87,0.04,Action (23.5%),PS3 (15.7%),2008
2,JP_Sales,0.10,2577,14021,0.45,0.01,Role-Playing (22.3%),PS2 (12.8%),2011
3,Global_Sales,1.08,1893,14705,2.91,0.23,Action (20.3%),PS2 (16.0%),2008


In [10]:
standard_market_df.shape

(14705, 11)

In [11]:
standard_market_df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
1893,1895,NASCAR Thunder 2004,PS2,2003.0,Racing,Electronic Arts,0.53,0.41,0.00,0.14,1.08
1894,1896,Prince of Persia,PS3,2008.0,Action,Ubisoft,0.47,0.41,0.03,0.18,1.08
1895,1897,SpongeBob SquarePants: Revenge of the Flying D...,PS2,2002.0,Platform,THQ,0.53,0.41,0.00,0.14,1.08
1896,1898,Grand Theft Auto V,PC,2015.0,Action,Take-Two Interactive,0.36,0.64,0.00,0.08,1.08
1897,1899,The Biggest Loser,Wii,2009.0,Sports,THQ,0.87,0.12,0.00,0.09,1.08


In [14]:
# 1. Keep your original columns but "melt" the sales regions into a single column
main_melted_df = df.melt(
    id_vars=['Name', 'Platform', 'Year', 'Genre'], # Columns to keep as they are
    value_vars=['NA_Sales', 'EU_Sales', 'JP_Sales', 'Global_Sales'], # Your sales columns
    var_name='Sales_Region', # This matches your summary table exactly!
    value_name='Sales_Amount'
)

# 2. Save this as your main data file
main_melted_df.to_excel("games_main_data_processed.xlsx", index=False)


In [13]:
main_melted_df.head()

,Name,Platform,Year,Genre,Sales_Region,Sales_Amount
0,Wii Sports,Wii,2006.0,Sports,NA_Sales,41.490002
1,Super Mario Bros.,NES,1985.0,Platform,NA_Sales,29.080000
2,Mario Kart Wii,Wii,2008.0,Racing,NA_Sales,15.850000
3,Wii Sports Resort,Wii,2009.0,Sports,NA_Sales,15.750000
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,NA_Sales,11.270000
